# BDC 2026 - Build Augmented Image Cache

Notebook ini cuma buat generate augmented images sekali, simpan ke disk + zip.
Nanti pas training tinggal load dari cache, gak perlu augmentasi ulang tiap epoch.

In [ ]:
import hashlib
import math
import random
import zipfile
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from torchvision import transforms
from tqdm.auto import tqdm

In [ ]:
PROJECT_ROOT = Path("/kaggle/input/datasets/user235162736/bdc-2026/BDC2026/BDC2026")
OUTPUT_DIR = Path("/kaggle/working")
TRAIN_DIR = PROJECT_ROOT / "train"

IMAGE_SIZE = 224
AUGS_PER_IMAGE = 4
SEED = 42

CACHE_FORMAT = "JPEG"
CACHE_QUALITY = 92
AUGMENTATION_VERSION = "pil_crop_flip_rotate_shift_color_blur_gray_cutout_v1"
PREPROCESSING_VERSION = "exif_rgb_fit_pad_v1"

AUG_IMAGE_CACHE_DIR = OUTPUT_DIR / "aug_image_cache"
SHARD_SIZE = 2000
MAX_SOURCE_DIM = 1024
MANIFEST_FLUSH_EVERY = 2000

VISION_MEAN = [0.485, 0.456, 0.406]
VISION_STD = [0.229, 0.224, 0.225]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def default_padding_from_mean(mean: list[float]) -> tuple[int, int, int]:
    return tuple(int(round(float(m) * 255)) for m in mean)


def stable_aug_seed(image_id: str, aug_idx: int, seed: int) -> int:
    payload = f"{image_id}|{aug_idx}|{seed}".encode("utf-8")
    return int(hashlib.sha256(payload).hexdigest()[:8], 16)


def get_cache_config_hash() -> str:
    payload = (
        f"{AUGMENTATION_VERSION}|{AUGS_PER_IMAGE}|{IMAGE_SIZE}|"
        f"{SEED}|{CACHE_FORMAT}|{CACHE_QUALITY}|{PREPROCESSING_VERSION}"
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:16]


def get_augmented_cache_dir() -> Path:
    return AUG_IMAGE_CACHE_DIR / get_cache_config_hash()


def is_augmented_cache_valid(expected_images: Optional[int] = None) -> bool:
    cache_dir = get_augmented_cache_dir()
    manifest_path = cache_dir / "manifest.csv"
    hash_path = cache_dir / "config_hash.txt"
    if not manifest_path.exists() or not hash_path.exists():
        return False
    if hash_path.read_text(encoding="utf-8").strip() != get_cache_config_hash():
        return False
    try:
        manifest = pd.read_csv(manifest_path)
    except Exception:
        return False
    if expected_images is not None and len(manifest) != expected_images * AUGS_PER_IMAGE:
        return False
    return True

In [ ]:
def downscale_if_huge(image: Image.Image, max_dim: int) -> Image.Image:
    width, height = image.size
    longest = max(width, height)
    if longest <= max_dim:
        return image
    scale = max_dim / longest
    new_w = max(1, int(round(width * scale)))
    new_h = max(1, int(round(height * scale)))
    return image.resize((new_w, new_h), Image.Resampling.BICUBIC)


def resize_and_pad(image: Image.Image, size: int, fill: tuple[int, int, int]) -> Image.Image:
    image = image.convert("RGB")
    width, height = image.size
    if width <= 0 or height <= 0:
        raise ValueError(f"Invalid image size: {image.size}")
    scale = min(size / width, size / height)
    new_w = max(1, int(round(width * scale)))
    new_h = max(1, int(round(height * scale)))
    resized = image.resize((new_w, new_h), Image.Resampling.BICUBIC)
    canvas = Image.new("RGB", (size, size), fill)
    canvas.paste(resized, ((size - new_w) // 2, (size - new_h) // 2))
    return canvas


def random_resized_crop_pil(image: Image.Image, rng: random.Random) -> Image.Image:
    width, height = image.size
    area = width * height
    for _ in range(10):
        target_area = area * rng.uniform(0.78, 1.0)
        aspect = math.exp(rng.uniform(math.log(0.85), math.log(1.18)))
        crop_w = int(round(math.sqrt(target_area * aspect)))
        crop_h = int(round(math.sqrt(target_area / aspect)))
        if 0 < crop_w <= width and 0 < crop_h <= height:
            left = rng.randint(0, width - crop_w)
            top = rng.randint(0, height - crop_h)
            return image.crop((left, top, left + crop_w, top + crop_h))
    side = min(width, height)
    left = (width - side) // 2
    top = (height - side) // 2
    return image.crop((left, top, left + side, top + side))


def apply_deterministic_augmentation(image: Image.Image, seed: int, fill: tuple[int, int, int]) -> Image.Image:
    rng = random.Random(seed)
    image = ImageOps.exif_transpose(image).convert("RGB")
    image = downscale_if_huge(image, MAX_SOURCE_DIM)

    image = random_resized_crop_pil(image, rng)
    if rng.random() < 0.5:
        image = ImageOps.mirror(image)
    angle = rng.uniform(-15.0, 15.0)
    image = image.rotate(angle, resample=Image.Resampling.BICUBIC, expand=False, fillcolor=fill)
    if rng.random() < 0.5:
        max_shift = 0.08
        dx = int(round(rng.uniform(-max_shift, max_shift) * image.size[0]))
        dy = int(round(rng.uniform(-max_shift, max_shift) * image.size[1]))
        image = image.transform(
            image.size,
            Image.AFFINE,
            (1, 0, dx, 0, 1, dy),
            resample=Image.Resampling.BICUBIC,
            fillcolor=fill,
        )

    image = ImageEnhance.Brightness(image).enhance(rng.uniform(0.80, 1.20))
    image = ImageEnhance.Contrast(image).enhance(rng.uniform(0.80, 1.20))
    image = ImageEnhance.Color(image).enhance(rng.uniform(0.80, 1.20))
    image = ImageEnhance.Sharpness(image).enhance(rng.uniform(0.85, 1.25))

    if rng.random() < 0.20:
        image = image.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.3, 1.1)))
    if rng.random() < 0.12:
        image = image.convert("L").convert("RGB")
    if rng.random() < 0.10:
        image = ImageOps.autocontrast(image, cutoff=int(rng.uniform(0, 2)))

    if rng.random() < 0.25:
        w, h = image.size
        ew = max(1, int(w * rng.uniform(0.08, 0.22)))
        eh = max(1, int(h * rng.uniform(0.08, 0.22)))
        ex = rng.randint(0, max(0, w - ew))
        ey = rng.randint(0, max(0, h - eh))
        image.paste(Image.new("RGB", (ew, eh), fill), (ex, ey))

    return image

## Load metadata

Scan folder `train/<label>_<class_name>/*.jpg` langsung, sesuai struktur dataset BDC2026 (`0_Recyclable`, `1_Electronic`, `2_Organic`).

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

records = []
for class_dir in sorted(TRAIN_DIR.iterdir()):
    if not class_dir.is_dir():
        continue
    label_str, class_name = class_dir.name.split("_", 1)
    label = int(label_str)
    img_paths = sorted(
        p for p in class_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    for img_path in img_paths:
        records.append({
            "image_id": f"{class_dir.name}/{img_path.name}",
            "image_path": str(img_path),
            "label": label,
            "class_name": class_name,
        })

train_df = pd.DataFrame.from_records(records)
print(f"{len(train_df)} training images across {train_df['class_name'].nunique()} classes")
train_df["class_name"].value_counts()

26527 training images across 3 classes


class_name
Organic       12567
Recyclable     9999
Electronic     3961
Name: count, dtype: int64

In [ ]:
def build_augmented_cache(df: pd.DataFrame) -> pd.DataFrame:
    """Sequential, sharded filenames avoid slow lookups once file count gets huge."""
    cache_dir = get_augmented_cache_dir()
    cache_dir.mkdir(parents=True, exist_ok=True)
    fill = default_padding_from_mean(VISION_MEAN)
    ext = CACHE_FORMAT.lower().replace("jpeg", "jpg")

    rows = []
    file_counter = 0
    for base_index, row in tqdm(df.reset_index(drop=True).iterrows(), total=len(df)):
        with Image.open(row.image_path) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            im = downscale_if_huge(im, MAX_SOURCE_DIM)
            im = im.copy()

        for aug_idx in range(AUGS_PER_IMAGE):
            seed = stable_aug_seed(str(row.image_id), aug_idx, SEED)
            aug_image = apply_deterministic_augmentation(im, seed, fill)
            aug_image = resize_and_pad(aug_image, IMAGE_SIZE, fill)

            shard = file_counter // SHARD_SIZE
            shard_dir = cache_dir / f"shard_{shard:04d}"
            shard_dir.mkdir(exist_ok=True)
            filename = f"aug_{file_counter:08d}.{ext}"
            fpath = shard_dir / filename
            file_counter += 1

            aug_image.save(fpath, format=CACHE_FORMAT, quality=CACHE_QUALITY)

            rows.append({
                "base_index": int(base_index),
                "image_id": str(row.image_id),
                "label": int(row.label),
                "class_name": str(row.class_name),
                "aug_idx": int(aug_idx),
                "feature_id": f"{row.image_id}::aug{aug_idx}",
                "cache_path": str(fpath.relative_to(OUTPUT_DIR)),
            })

        if file_counter % MANIFEST_FLUSH_EVERY < AUGS_PER_IMAGE:
            pd.DataFrame(rows).to_csv(cache_dir / "manifest.csv", index=False)

    manifest = pd.DataFrame(rows)
    manifest.to_csv(cache_dir / "manifest.csv", index=False)
    (cache_dir / "config_hash.txt").write_text(get_cache_config_hash(), encoding="utf-8")
    return manifest

In [ ]:
if is_augmented_cache_valid(expected_images=len(train_df)):
    print(f"Cache valid, skip building: {get_augmented_cache_dir()}")
    manifest = pd.read_csv(get_augmented_cache_dir() / "manifest.csv")
else:
    manifest = build_augmented_cache(train_df)

print(f"Cache dir: {get_augmented_cache_dir()}")
print(f"Manifest rows: {len(manifest)}")
manifest.head()

  0%|          | 0/26527 [00:00<?, ?it/s]

Cache dir: /kaggle/working/aug_image_cache/8315c7d99fb13736
Manifest rows: 106108


,base_index,image_id,label,class_name,aug_idx,feature_id,cache_path
0,0,0_Recyclable/R_1.jpg,0,Recyclable,0,0_Recyclable/R_1.jpg::aug0,aug_image_cache/8315c7d99fb13736/shard_0000/au...
1,0,0_Recyclable/R_1.jpg,0,Recyclable,1,0_Recyclable/R_1.jpg::aug1,aug_image_cache/8315c7d99fb13736/shard_0000/au...
2,0,0_Recyclable/R_1.jpg,0,Recyclable,2,0_Recyclable/R_1.jpg::aug2,aug_image_cache/8315c7d99fb13736/shard_0000/au...
3,0,0_Recyclable/R_1.jpg,0,Recyclable,3,0_Recyclable/R_1.jpg::aug3,aug_image_cache/8315c7d99fb13736/shard_0000/au...
4,1,0_Recyclable/R_10.jpg,0,Recyclable,0,0_Recyclable/R_10.jpg::aug0,aug_image_cache/8315c7d99fb13736/shard_0000/au...


## Zip cache

Struktur di dalam zip dijaga relatif ke `OUTPUT_DIR`, jadi tinggal `extractall(OUTPUT_DIR)` pas dipakai lagi.

In [ ]:
def zip_augmented_cache() -> Path:
    cache_dir = get_augmented_cache_dir()
    zip_path = OUTPUT_DIR / "aug_image_cache.zip"
    files = list(cache_dir.rglob("*"))
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for f in tqdm(files):
            if f.is_file():
                archive.write(f, f.relative_to(OUTPUT_DIR))
    return zip_path


zip_path = zip_augmented_cache()
size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f"Saved: {zip_path} ({size_mb:.1f} MB)")

  0%|          | 0/106164 [00:00<?, ?it/s]

Saved: /kaggle/working/aug_image_cache.zip (1491.4 MB)


## Cara pakai pas training

```python
with zipfile.ZipFile("aug_image_cache.zip", "r") as archive:
    archive.extractall(OUTPUT_DIR)

manifest = pd.read_csv(get_augmented_cache_dir() / "manifest.csv")
```

Lalu di `Dataset.__getitem__`, pilih baris manifest sesuai `image_id` + `aug_idx` (random tiap epoch atau fixed), load `cache_path` dengan `Image.open`, langsung `to_tensor` + `normalize` - gak perlu augmentasi ulang.